# Part 2 - Your First Multi-Agent System with CrewAI

*Last updated: November 2026*

Two agents with different **roles, goals and backstories** work together:
a **chef** writes five recipes from your ingredients, and a **nutritionist** reviews them and recommends
the best one for diabetic patients. You will learn how **Agents**, **Tasks** and a **Crew** fit together and
how `{ingredients}` placeholders are filled from `kickoff(inputs=...)`.

## How to run this notebook

1. Set up the course environment once, in the course folder: `uv sync`.
   New to coding? Follow `BEGINNER_SETUP.md` step by step. Otherwise see `LOCAL_SETUP.MD`.
2. Select the **`.venv`** Python kernel (*Python 3.12*) for this notebook.
3. Copy `.env.example` (course folder) to `.env` and add your keys. The notebook finds `.env` automatically.

**Keys and settings used in this notebook**
- `OPENAI_API_KEY` (required)

Run the cells from top to bottom.

> **Compatibility update (differs from the recording)**
> - The recorded CrewAI 0.28 can no longer be installed: its tools package depends on `lancedb` versions that were removed from PyPI.
>   This notebook uses **CrewAI 1.x**. Agents, tasks, crews and `{placeholders}` work the same way.
> - `verbose=2` is now `verbose=True`, and the console output looks different from the video.
> - The model is `gpt-4o-mini` instead of `gpt-4`, which is far more expensive.

In [ ]:
# Setup check: run this cell first.
# Nothing is installed here. All packages were installed once with `uv sync` (see LOCAL_SETUP.MD).
import sys
from pathlib import Path

python_version = sys.version.split()[0]

if sys.version_info[:2] != (3, 12):
    print(f"Warning: this course is tested on Python 3.12, but this kernel runs Python {python_version}.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
elif Path(sys.prefix).name != ".venv":
    print("Warning: this kernel is not the course .venv environment, so some packages may be missing.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
else:
    print("Ready: course .venv kernel | Python", python_version)

In [ ]:
# Load API keys from the .env file, so keys are never written in the notebook.
# load_dotenv() looks for .env in this folder and then in each parent folder (the course folder).
import os

from dotenv import load_dotenv

load_dotenv()

REQUIRED_KEYS = ['OPENAI_API_KEY']

missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]
if missing:
    print("Missing required keys:", ", ".join(missing), "- see 'How to run this notebook' above.")
else:
    print("All required API keys are loaded.")

In [ ]:
# Hide library deprecation warnings so the lesson output stays readable
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from crewai import Agent, Task, Crew

In [ ]:
import os

# Model used by every agent in this crew
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

In [ ]:
# 1 - Chef
chef = Agent(
    role="chef",
    goal="Give 5 different receipes using ingredients : {ingredients}",
    backstory="You're  a Chef who has experience on cooking delicious foods"
              "Your expertice lies in recommeding some new foods to the customers"
              "Please generate and give receipe of 5 different foods using this ingredients : {ingredients} "
              "describing stepwise process, time to cook, other ingredients needed and calorie count"
              "Your recipe will be critically analyzed by nutritutionist to recommend food for diabetes patients ",
    verbose=True
)

In [ ]:
# 2 - Nutritionist
nutritionist = Agent(
    role="nutritionist",
    goal="Critically analyze the recipe given by chef using these ingredients: {ingredients}",
    backstory="You're a nutritionist and have good knowledge on working with Diabetic patients "
              "There are recipe recommended by chef using these ingredients : {ingredients}. "
              "please critically analyze the recipe and suggest best food out of goven 5 for Diabetic patients"
              "Also give reasons why you dont recommend other 4 receips and choose this one",
    verbose=True
)

In [ ]:
cook = Task(
    description=(
        "1. Write receipe of 5 different foods using these ingredients  : {ingredients}.\n"
    ),
    expected_output="5 different receipes using {ingredients}"
                    "Stepwise guide to cook"
                    "List of all additional ingredients required for cooking"
                    "The expected taste of the food",
    agent=chef,
)

In [ ]:
recommend = Task(
    description=(
        "1. Read the receipes provided by chef using {ingredients} \n"
        "2. critically analyze all receipes keeping in mind what should suit to diabetic patients.\n"
        "3. Suggest one best food out of all 5 receipes given by chef.\n"
        "4. Prove that other 4 receipes suggested by chef is not good for diabetic patients.\n"
    ),
    expected_output="A well-written critical pointwise analysis of all 5 recipes using {ingredients}"
                    "Suggest one receipe to diabetic patients based on your own knowledge and evidence "
                    "Reject all other 4 receipes with knowledge and evidence "
                    "Finally, give the stepwoise receipe of what you selected and a nice message",
    agent=nutritionist,
)

In [ ]:
crewaman = Crew(
    agents=[chef, nutritionist],
    tasks=[cook, recommend],
    verbose=True
)

In [ ]:
result = crewaman.kickoff(inputs={"ingredients": "chicken"})

In [ ]:
# Final answer from the last task (the nutritionist's recommendation)
print(result.raw)